In [4]:
import pandas as pd
import numpy as np
import optuna

import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

c:\Users\Damien\.conda\envs\DS4SC\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import des données

In [5]:
df = pd.read_csv("../../data/resample_normalized_flagged_v3.csv")

## Préparation des données

In [6]:
df = df.drop(columns=["apogee_id", "STARFLAGS"])

class_columns = ['class_spectral', 'class_lum_logg', 'class_lum_jhk', 'class_lum_bins_logg', 'class_lum_bins_jhk']

chemical_columns = ['C_FE', 'CI_FE', 'N_FE', 'O_FE', 'NA_FE', 'MG_FE', 'AL_FE', 'SI_FE', 'S_FE', 'K_FE', 'CA_FE', 'TI_FE', 'V_FE', 'CR_FE', 'MN_FE', 'NI_FE', 'FE_H']
physique_columns = ['J', 'H', 'K', 'LOGG', 'M_H', 'VMICRO', 'VMACRO']

df_chem = df.drop(columns=physique_columns)
df_phys = df.drop(columns=chemical_columns)

In [7]:
target_column = 'class_lum_logg'
X = df_chem.drop(columns=class_columns)
y = df_chem[target_column]

In [8]:
# Calcul des fréquences des classes
frequencies = df_chem[target_column].value_counts(normalize=True)

# Calcul des poids inverses
weights = (1 / frequencies).to_dict()

In [9]:
frequencies

class_lum_logg
III    0.513275
V      0.322261
IV     0.096601
II     0.057103
Ib     0.010761
Name: proportion, dtype: float64

### Implémentation de LightGBM

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_LGB, X_test_LGB, y_train_LGB, y_test_LGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, class_weight=weights, random_state=42, objective="multiclass", num_class=3)

lgb_model.fit(X_train_LGB, y_train_LGB, eval_set=[(X_test_LGB, y_test_LGB)])

y_pred_LGB = lgb_model.predict(X_test_LGB)

In [ ]:
#évaluation du modèle
print("Accuracy : \n", accuracy_score(y_test_LGB, y_pred_LGB))
print("Classification Report : \n", classification_report(y_test_LGB, y_pred_LGB))

In [ ]:
f1_LGB = f1_score(y_test_LGB, y_pred_LGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_LGB)

In [ ]:
y_probs_LGB = lgb_model.predict_proba(X_test_LGB)  # Probabilités prédites des classes

auc_pr_list_LGB = []
for idx, class_name in enumerate(np.unique(y_test_LGB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_LGB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_LGB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_LGB.append(auc_pr)
auc_pr_LGB = np.mean(auc_pr_list_LGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_LGB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_LGB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

### Implémentation de HistGradientBoosting

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_HGB, X_test_HGB, y_train_HGB, y_test_HGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sample_weights = compute_sample_weight(class_weight=weights, y=y_train_HGB)

# Initialisation du modèle
hgb = HistGradientBoostingClassifier(loss="log_loss", learning_rate=0.1, max_iter=100)

# Entraînement
hgb.fit(X_train_HGB, y_train_HGB, sample_weight=sample_weights)

# Prédiction
y_pred_HGB = hgb.predict(X_test_HGB)

In [ ]:
# Évaluation
print("Accuracy:", accuracy_score(y_test_HGB, y_pred_HGB))
print("Classification Report:\n", classification_report(y_test_HGB, y_pred_HGB))

In [ ]:
f1_HGB = f1_score(y_test_HGB, y_pred_HGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_HGB)

In [ ]:
y_probs_HGB = hgb.predict_proba(X_test_HGB)  # Probabilités prédites des classes

auc_pr_list_HGB = []
for idx, class_name in enumerate(np.unique(y_test_HGB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_HGB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_HGB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_HGB.append(auc_pr)
auc_pr_HGB = np.mean(auc_pr_list_HGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_HGB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_HGB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

### Implémentation de CatBoost

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_CB, X_test_CB, y_train_CB, y_test_CB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Création et entraînement du modèle
model_catboost = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, loss_function='MultiClass',  eval_metric='MultiClass', class_weights=weights, verbose=100)

model_catboost.fit(X_train_CB, y_train_CB, eval_set=(X_test_CB, y_test_CB), early_stopping_rounds=100)

# Prédictions
y_pred = model_catboost.predict(X_test_CB)

In [ ]:
# Performance
print("Accuracy:", accuracy_score(y_test_CB, y_pred))
print(classification_report(y_test_CB, y_pred))

In [ ]:
f1_CB = f1_score(y_test_CB, y_pred, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_CB)

In [ ]:
y_probs_CB = model_catboost.predict_proba(X_test_CB)  # Probabilités prédites des classes

auc_pr_list_CB = []
for idx, class_name in enumerate(np.unique(y_test_CB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_CB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_CB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_CB.append(auc_pr)
auc_pr_CB = np.mean(auc_pr_list_CB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_CB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_CB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

## Optimisation des hyperparamètres avec Optuna

### opti LightGBM

In [12]:
# Division en ensemble d'entraînement et de test
X_train_LGB, X_test_LGB, y_train_LGB, y_test_LGB = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

def objective_LGB(trial):
    params = {
        'objective': 'multiclass',
        'metric': 'multi_logloss',
        'num_class': 3,
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 16),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
    }

    lgb_model = lgb.LGBMClassifier(**params, class_weight=weights, random_state=42)
    lgb_model.fit(X_train_LGB, 
              y_train_LGB, 
              eval_set=[(X_test_LGB, y_test_LGB)], 
              )

    y_pred = lgb_model.predict(X_test_LGB)
    y_probs_LGB = lgb_model.predict_proba(X_test_LGB)

    f1 = f1_score(y_test_LGB, y_pred, average='weighted')
    auc_pr_list_LGB = []
    for idx, class_name in enumerate(np.unique(y_test_LGB)):
        # Créer une version binaire de y_test_LGB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_LGB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_LGB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_LGB.append(auc_pr)
    auc_pr_LGB = np.mean(auc_pr_list_LGB)  # Moyenne sur toutes les classes


    return f1 , auc_pr_LGB

# Créer un objet d'étude Optuna
study_LGB = optuna.create_study(directions=['maximize', 'maximize'])
# Optimiser la fonction objectif
study_LGB.optimize(objective_LGB, n_trials=50, show_progress_bar=True)

[I 2025-03-30 01:49:17,974] A new study created in memory with name: no-name-2daa4639-fbad-49e1-81f3-d577814ab6d2
  0%|          | 0/50 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003161 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 317766, number of used features: 17
[LightGBM] [Info] Start training from score -1.609482
[LightGBM] [Info] Start training from score -1.609463
[LightGBM] [Info] Start training from score -1.609473
[LightGBM] [Info] Start training from score -1.609314
[LightGBM] [Info] Start training from score -1.609457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

  2%|▏         | 1/50 [02:14<1:49:38, 134.26s/it]

[I 2025-03-30 01:51:32,236] Trial 0 finished with values: [0.8554784032875719, 0.8599458817850401] and parameters: {'num_leaves': 159, 'max_depth': 5, 'learning_rate': 0.014211720999949485, 'n_estimators': 1513}.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013420 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 317766, number of used features: 17
[LightGBM] [Info] Start training from score -1.609482
[LightGBM] [Info] Start training from score -1.609463
[LightGBM] [Info] Start training from score -1.609473
[LightGBM] [Info] Start training from score -1.609314
[LightGBM] [Info] Start training from score -1.609457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

  4%|▍         | 2/50 [03:59<1:33:41, 117.12s/it]

...

[I 2025-03-30 04:21:56,002] Trial 47 finished with values: [0.8715555684555378, 0.8723059752458575] and parameters: {'num_leaves': 111, 'max_depth': 3, 'learning_rate': 0.13279211558844622, 'n_estimators': 1844}.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013559 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 317766, number of used features: 17
[LightGBM] [Info] Start training from score -1.609482
[LightGBM] [Info] Start training from score -1.609463
[LightGBM] [Info] Start training from score -1.609473
[LightGBM] [Info] Start training from score -1.609314
[LightGBM] [Info] Start training from score -1.609457


 98%|█████████▊| 49/50 [1:34:33<02:08, 128.04s/it]

[I 2025-03-30 04:23:51,446] Trial 48 finished with values: [0.9037156343019885, 0.8955528877698962] and parameters: {'num_leaves': 32, 'max_depth': 15, 'learning_rate': 0.14454327296893707, 'n_estimators': 1679}.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013904 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 317766, number of used features: 17
[LightGBM] [Info] Start training from score -1.609482
[LightGBM] [Info] Start training from score -1.609463
[LightGBM] [Info] Start training from score -1.609473
[LightGBM] [Info] Start training from score -1.609314
[LightGBM] [Info] Start training from score -1.609457


100%|██████████| 50/50 [1:36:59<00:00, 116.39s/it]

[I 2025-03-30 04:26:17,551] Trial 49 finished with values: [0.8932313085326659, 0.889867906056752] and parameters: {'num_leaves': 34, 'max_depth': 6, 'learning_rate': 0.07366254549293658, 'n_estimators': 1732}.


In [13]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_LGB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_LGB.best_trials[0].params)
print("Meilleur score F1 : ", study_LGB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_LGB.best_trials[0].values[1])

Meilleur essai :  FrozenTrial(number=10, state=1, values=[0.9091984212095777, 0.9033095705304595], datetime_start=datetime.datetime(2025, 3, 30, 3, 3, 56, 252052), datetime_complete=datetime.datetime(2025, 3, 30, 3, 8, 3, 194666), params={'num_leaves': 132, 'max_depth': 14, 'learning_rate': 0.11563518283576125, 'n_estimators': 1948}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributions={'num_leaves': IntDistribution(high=255, log=False, low=31, step=1), 'max_depth': IntDistribution(high=16, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.2, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=2000, log=False, low=100, step=1)}, trial_id=10, value=None)
Meilleurs hyperparamètres :  {'num_leaves': 132, 'max_depth': 14, 'learning_rate': 0.11563518283576125, 'n_estimators': 1948}
Meilleur score F1 :  0.9091984212095777
Meilleur score AUC-PR :  0.9033095705304595


### opti HistGradientBoosting

In [14]:
# Division des données
X_train_HGB, X_test_HGB, y_train_HGB, y_test_HGB = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Poids des classes
sample_weights_HGB = compute_sample_weight(class_weight="balanced", y=y_train_HGB)

def objective_HGB(trial):
    """ Fonction d'optimisation pour Optuna """

    # Hyperparamètres à optimiser
    params = {
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        "max_iter": trial.suggest_int("max_iter", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)
    }

    # Initialisation du modèle
    model_HGB = HistGradientBoostingClassifier(loss="log_loss", **params, random_state=42)

    # Entraînement avec les poids
    model_HGB.fit(X_train_HGB, y_train_HGB, sample_weight=sample_weights_HGB)

    # Prédiction
    y_pred_HGB = model_HGB.predict(X_test_HGB)
    y_probs_HGB = model_HGB.predict_proba(X_test_HGB)

    # Calcul du F1-score pondéré (mieux adapté au déséquilibre)
    score = f1_score(y_test_HGB, y_pred_HGB, average="weighted")
    # Calcul l'AUC-PR
    auc_pr_list_HGB = []
    for idx, class_name in enumerate(np.unique(y_test_HGB)):
        # Créer une version binaire de y_test_HGB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_HGB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_HGB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_HGB.append(auc_pr)
    auc_pr_HGB = np.mean(auc_pr_list_HGB)  # Moyenne sur toutes les classes

    return score, auc_pr_HGB  # Optuna va maximiser ces valeurs

# Lancer l'optimisation
study_HGB = optuna.create_study(directions=['maximize', 'maximize'])
study_HGB.optimize(objective_HGB, n_trials=50, show_progress_bar=True)

[I 2025-03-30 04:26:18,249] A new study created in memory with name: no-name-5f8a6915-1ef7-48d9-b4a1-cac62363958d
  0%|          | 0/50 [00:00<?, ?it/s]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)
  2%|▏         | 1/50 [00:13<10:58, 13.44s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:12: FutureWarning: suggest_loguniform 

[I 2025-03-30 04:26:31,690] Trial 0 finished with values: [0.8603092942671123, 0.857853376759804] and parameters: {'learning_rate': 0.14627648123790027, 'max_iter': 300, 'max_depth': 9, 'min_samples_leaf': 46, 'l2_regularization': 0.00012268992965006132}.


  4%|▍         | 2/50 [00:45<19:19, 24.16s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)


[I 2025-03-30 04:27:03,353] Trial 1 finished with values: [0.8624131537138909, 0.8691659238429625] and parameters: {'learning_rate': 0.058293449762788954, 'max_iter': 700, 'max_depth': 8, 'min_samples_leaf': 36, 'l2_regularization': 0.00015835848083631238}.


  6%|▌         | 3/50 [02:04<38:37, 49.30s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)


...

[I 2025-03-30 04:54:44,928] Trial 47 finished with values: [0.8614740017526848, 0.8652846029021297] and parameters: {'learning_rate': 0.05762969991392817, 'max_iter': 800, 'max_depth': 5, 'min_samples_leaf': 70, 'l2_regularization': 0.025017236326039378}.


 98%|█████████▊| 49/50 [28:41<00:21, 21.47s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\2278091007.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)


[I 2025-03-30 04:55:00,222] Trial 48 finished with values: [0.8653285730818843, 0.8689036231291605] and parameters: {'learning_rate': 0.14673641886442856, 'max_iter': 200, 'max_depth': 9, 'min_samples_leaf': 85, 'l2_regularization': 0.00011603956040303694}.


100%|██████████| 50/50 [28:51<00:00, 34.63s/it]

[I 2025-03-30 04:55:09,881] Trial 49 finished with values: [0.7525568347838202, 0.7332390919090037] and parameters: {'learning_rate': 0.01572443938142459, 'max_iter': 100, 'max_depth': 4, 'min_samples_leaf': 60, 'l2_regularization': 0.07271458737673618}.


In [15]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_HGB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_HGB.best_trials[0].params)
print("Meilleur score F1 : ", study_HGB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_HGB.best_trials[0].values[1])

Meilleur essai :  FrozenTrial(number=19, state=1, values=[0.8649534834190284, 0.8701291254264547], datetime_start=datetime.datetime(2025, 3, 30, 4, 37, 41, 120280), datetime_complete=datetime.datetime(2025, 3, 30, 4, 38, 10, 393984), params={'learning_rate': 0.07114032584587605, 'max_iter': 800, 'max_depth': 7, 'min_samples_leaf': 44, 'l2_regularization': 0.0028018053446478798}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None), 'max_iter': IntDistribution(high=1000, log=False, low=100, step=100), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_samples_leaf': IntDistribution(high=100, log=False, low=10, step=1), 'l2_regularization': FloatDistribution(high=0.1, log=True, low=0.0001, step=None)}, trial_id=19, value=None)
Meilleurs hyperparamètres :  {'learning_rate': 0.07114032584587605, 'max_iter': 800, 'max_depth': 7, 'min_samples_leaf': 44, 'l2_re

### opti CatBoost

In [16]:
# Division des données
X_train_CB, X_test_CB, y_train_CB, y_test_CB = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Poids des classes pour gérer le déséquilibre
sample_weights_CB = compute_sample_weight(class_weight="balanced", y=y_train_CB)

def objective_CB(trial):
    """ Fonction d'optimisation pour Optuna """

    # Hyperparamètres à optimiser
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000, step=250),
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        "depth": trial.suggest_int("depth", 4, 12),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),
    }

    # Création et entraînement du modèle
    model_CB = CatBoostClassifier(
        **params, 
        loss_function="MultiClass", 
        eval_metric="MultiClass", 
        verbose=0, 
        random_seed=42
    )

    model_CB.fit(X_train_CB, y_train_CB, sample_weight=sample_weights_CB, eval_set=(X_test_CB, y_test_CB), early_stopping_rounds=100, verbose=False)


    # Prédiction
    y_pred_CB = model_CB.predict(X_test_HGB)
    y_probs_CB = model_CB.predict_proba(X_test_HGB)

    # Calcul du F1-score pondéré (mieux adapté au déséquilibre)
    score = f1_score(y_test_CB, y_pred_CB, average="weighted")
    # Calcul l'AUC-PR
    auc_pr_list_CB = []
    for idx, class_name in enumerate(np.unique(y_test_CB)):
        # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_CB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_CB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_CB.append(auc_pr)
    auc_pr_CB = np.mean(auc_pr_list_CB)  # Moyenne sur toutes les classes

    return score, auc_pr_CB  # Optuna va maximiser ces valeurs

# Lancer l'optimisation
study_CB = optuna.create_study(directions=['maximize', 'maximize'])
study_CB.optimize(objective_CB, n_trials=50, show_progress_bar=True)

[I 2025-03-30 04:55:10,500] A new study created in memory with name: no-name-e9b73967-b85d-4a31-90af-57268f36767b
  0%|          | 0/50 [00:00<?, ?it/s]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:16: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),
  2%|▏         | 1/50 [01:40<1:22:22, 100.87s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:13: FutureWarning: suggest_loguniform has been deprecated

[I 2025-03-30 04:56:51,374] Trial 0 finished with values: [0.8430878688706989, 0.8405594013476391] and parameters: {'iterations': 1250, 'learning_rate': 0.03410263171510557, 'depth': 5, 'border_count': 167, 'bagging_temperature': 0.8030464004939908}.


  4%|▍         | 2/50 [05:41<2:26:27, 183.08s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:16: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),


[I 2025-03-30 05:00:51,994] Trial 1 finished with values: [0.8877150100023856, 0.8858510172592707] and parameters: {'iterations': 1750, 'learning_rate': 0.18654951561602903, 'depth': 7, 'border_count': 93, 'bagging_temperature': 0.8491622082701883}.


  6%|▌         | 3/50 [27:45<9:11:37, 704.20s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:16: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),


...

[I 2025-03-30 11:15:32,684] Trial 47 finished with values: [0.8439862438799401, 0.844115498136596] and parameters: {'iterations': 500, 'learning_rate': 0.024144249023296635, 'depth': 10, 'border_count': 229, 'bagging_temperature': 0.9287023427678257}.


 98%|█████████▊| 49/50 [6:22:21<05:32, 332.36s/it]C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
C:\Users\Damien\AppData\Local\Temp\ipykernel_7704\3126877657.py:16: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),


[I 2025-03-30 11:17:31,621] Trial 48 finished with values: [0.8584722825337976, 0.8614245745601611] and parameters: {'iterations': 1500, 'learning_rate': 0.06364799414049807, 'depth': 5, 'border_count': 104, 'bagging_temperature': 0.20065567145835106}.


100%|██████████| 50/50 [6:22:58<00:00, 459.57s/it]

[I 2025-03-30 11:18:09,027] Trial 49 finished with values: [0.8288061857520341, 0.8140206789744446] and parameters: {'iterations': 500, 'learning_rate': 0.059882914266208016, 'depth': 4, 'border_count': 239, 'bagging_temperature': 0.4363492693071712}.


In [17]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_CB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_CB.best_trials[0].params)
print("Meilleur score F1 : ", study_CB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_CB.best_trials[0].values[1])

Meilleur essai :  FrozenTrial(number=33, state=1, values=[0.9033585928527593, 0.8963260277111592], datetime_start=datetime.datetime(2025, 3, 30, 8, 51, 46, 813614), datetime_complete=datetime.datetime(2025, 3, 30, 9, 27, 4, 541961), params={'iterations': 2000, 'learning_rate': 0.11142941674272973, 'depth': 12, 'border_count': 143, 'bagging_temperature': 0.4737865740129352}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributions={'iterations': IntDistribution(high=2000, log=False, low=500, step=250), 'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None), 'depth': IntDistribution(high=12, log=False, low=4, step=1), 'border_count': IntDistribution(high=255, log=False, low=32, step=1), 'bagging_temperature': FloatDistribution(high=1.0, log=False, low=0.0, step=None)}, trial_id=33, value=None)
Meilleurs hyperparamètres :  {'iterations': 2000, 'learning_rate': 0.11142941674272973, 'depth': 12, 'border_count': 143, 'bagging_temperature